# 10. Auditoria del flujo LLM de produccion con prompt v3 final

Este notebook audita el flujo usado por la aplicacion de triaje a partir de historias clinicas libres. El objetivo no es entrenar ni seleccionar un modelo nuevo, sino comprobar de forma trazable que el extractor LLM, el `VectorClinico`, el adaptador y el predictor final se encadenan correctamente.

El LLM solo actua como extractor estructurado. La prediccion ESI la realiza el modelo LightGBM final congelado. Este prototipo no esta validado para uso clinico real.

## 1. Configuracion reproducible

Se usan rutas del paquete (`config.py`) y el prompt versionado `extractor_system_v3_final.txt`. No se leen secretos ni archivos `.env`.

In [1]:
import hashlib
import json
import time

import ollama
import pandas as pd
from IPython.display import Markdown, display

from triaje_ia.config import MODELS_DIR, PROJECT_ROOT, PROMPTS_DIR
from triaje_ia.inference.adapter import vectorclinico_a_features
from triaje_ia.inference.predictor import TriajePredictor
from triaje_ia.llm.extractor import SYSTEM_PROMPT, _completar_datos_explicitos
from triaje_ia.llm.normalizer import normalizar_vector_clinico
from triaje_ia.llm.schemas import VectorClinico
from triaje_ia.llm.validator import resumen_validacion, validar_vector_clinico

PROMPT_PATH = PROMPTS_DIR / "extractor_system_v3_final.txt"
LLM_MODEL = "llama3.1:8b-instruct-q4_K_M"

assert PROMPT_PATH.exists(), f"No existe el prompt esperado: {PROMPT_PATH}"
assert SYSTEM_PROMPT == PROMPT_PATH.read_text(encoding="utf-8")

prompt_sha256 = hashlib.sha256(SYSTEM_PROMPT.encode("utf-8")).hexdigest()
pd.DataFrame([
    {"elemento": "project_root", "valor": str(PROJECT_ROOT)},
    {"elemento": "prompt", "valor": PROMPT_PATH.name},
    {"elemento": "prompt_sha256", "valor": prompt_sha256},
    {"elemento": "modelo_llm", "valor": LLM_MODEL},
    {"elemento": "active_model", "valor": str(MODELS_DIR / "active_model.json")},
])

c:\Users\CARLOS\triaje-ia-tfg\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,elemento,valor
0,project_root,C:\Users\CARLOS\triaje-ia-tfg
1,prompt,extractor_system_v3_final.txt
2,prompt_sha256,aa834ac9762fcd0d0fda525a9b20943edc9718226c059e...
3,modelo_llm,llama3.1:8b-instruct-q4_K_M
4,active_model,C:\Users\CARLOS\triaje-ia-tfg\models\active_mo...


## 2. Carga del modelo final

El predictor se carga exactamente desde `models/active_model.json`. Esta celda no entrena, no recalibra y no modifica umbrales.

In [2]:
predictor = TriajePredictor()

pd.DataFrame([
    {"elemento": "n_features_modelo", "valor": len(predictor._feature_names)},
    {"elemento": "politica_decision", "valor": predictor._decision_policy},
    {"elemento": "umbral_alerta_a1", "valor": predictor._warning_threshold_a1},
    {"elemento": "n_features_tabulares", "valor": len(predictor._tabular_features)},
    {"elemento": "n_features_bert_svd", "valor": len(predictor._bert_cols)},
])

,elemento,valor
0,n_features_modelo,79
1,politica_decision,argmax_with_a1_warning
2,umbral_alerta_a1,0.4
3,n_features_tabulares,64
4,n_features_bert_svd,15


## 3. Casos de auditoria

Se definen 20 historias clinicas sintenticas controladas. Si existe un archivo local anonimo en `data/interim/llm_audit_real_cases_anonymized.csv`, se sustituyen hasta 5 casos sintenticos por casos reales anonimizados. Ese archivo debe contener, como minimo, una columna `narrativa`.

In [3]:
CASOS_SINTETICOS = [
    {"id": "S01", "tipo": "sintetico", "categoria": "critico_neuro", "narrativa": "Varon 67a. HTA y FA cronica. Tto bisoprolol y acenocumarol. Traido por familia por bajo nivel de consciencia brusco hace 1h. Sin fiebre. TA 185/110, FC 48 irregular, FR 14, SatO2 94%, T 36.2. GCS 10."},
    {"id": "S02", "tipo": "sintetico", "categoria": "dolor_toracico", "narrativa": "Mujer 72a con diabetes e hipertension. Dolor toracico opresivo de 45 min con sudoracion y nauseas. TA 92/58, FC 118, FR 24, SatO2 91%, T 36.8. Dolor 9/10."},
    {"id": "S03", "tipo": "sintetico", "categoria": "disnea", "narrativa": "Varon 64a EPOC. Consulta por disnea progresiva desde ayer y tos. TA 145/82, FC 108, FR 30, SatO2 86%, T 37.9. No dolor toracico."},
    {"id": "S04", "tipo": "sintetico", "categoria": "sepsis", "narrativa": "Mujer 81a con insuficiencia renal. Fiebre y deterioro general desde hace 2 dias, confusion desde esta manana. TA 88/52, FC 124, FR 28, SatO2 92%, T 39.1."},
    {"id": "S05", "tipo": "sintetico", "categoria": "cefalea", "narrativa": "Hombre de unos 40 anos con cefalea brusca, la peor de su vida, desde hace 30 min. Vomitos y fotofobia. TA 160/95, FC 96, FR 18, SatO2 98%, T 36.7."},
    {"id": "S06", "tipo": "sintetico", "categoria": "abdomen", "narrativa": "Mujer 52a dolor hipocondrio derecho desde la cena, nauseas. TA 134/78, FC 92, FR 18, SatO2 98%, T 37.4, Glasgow 15. EVA 6/10."},
    {"id": "S07", "tipo": "sintetico", "categoria": "abdomen_fid", "narrativa": "Varon 29a dolor en fosa iliaca derecha de 10 horas de evolucion, vomitos en dos ocasiones. TA 122/74, FC 101, FR 18, SatO2 99%, T 38.0. Dolor 7/10."},
    {"id": "S08", "tipo": "sintetico", "categoria": "sincope", "narrativa": "Mujer 58a sincope en domicilio con recuperacion completa. Mareo previo. HTA. TA 110/70, FC 54, FR 16, SatO2 97%, T 36.4. Niega dolor toracico."},
    {"id": "S09", "tipo": "sintetico", "categoria": "trauma", "narrativa": "Varon 35a caida de bicicleta hace 1h, dolor intenso en muneca derecha y herida superficial en rodilla. TA 128/80, FC 88, FR 16, SatO2 99%, T 36.6. Dolor 8/10."},
    {"id": "S10", "tipo": "sintetico", "categoria": "alergia", "narrativa": "Mujer 24a urticaria generalizada tras comer frutos secos, sensacion de garganta cerrada. TA 100/65, FC 120, FR 26, SatO2 95%, T 36.5."},
    {"id": "S11", "tipo": "sintetico", "categoria": "psiquiatrico", "narrativa": "Varon 46a traido por familia por ideacion autolitica. Ansiedad intensa, sin lesiones. TA 136/84, FC 112, FR 20, SatO2 98%, T 36.7."},
    {"id": "S12", "tipo": "sintetico", "categoria": "rectorragia", "narrativa": "Mujer 69a anticoagulada con acenocumarol por FA. Rectorragia desde ayer y debilidad. TA 98/60, FC 116, FR 20, SatO2 97%, T 36.3."},
    {"id": "S13", "tipo": "sintetico", "categoria": "convulsion", "narrativa": "Varon 31a epilepsia conocida, convulsion tonico-clonica hace 20 min, ahora somnoliento. TA 130/82, FC 105, FR 18, SatO2 96%, T 36.8."},
    {"id": "S14", "tipo": "sintetico", "categoria": "fiebre_leve", "narrativa": "Mujer 36a fiebre, tos y mialgias desde ayer. Sin disnea. TA 118/72, FC 98, FR 18, SatO2 98%, T 38.4. Dolor 3/10."},
    {"id": "S15", "tipo": "sintetico", "categoria": "otalgia", "narrativa": "Mujer 34a dolor de oido derecho desde ayer, febricula. Sin dolor dental. TA 122/70, FC 84, FR 16, SatO2 99%, T 37.5."},
    {"id": "S16", "tipo": "sintetico", "categoria": "dental", "narrativa": "Varon 45a dolor dental en molar inferior desde hace 3 dias. No fiebre. TA 130/80, FC 86, FR 16, SatO2 99%, T 36.7. Dolor 6/10."},
    {"id": "S17", "tipo": "sintetico", "categoria": "una_encarnada", "narrativa": "Varon 41a dolor en dedo gordo del pie por una una encarnada, rojo y algo inflamado. No fiebre. TA 128/76, FC 82, FR 16, SatO2 98%, T 36.7."},
    {"id": "S18", "tipo": "sintetico", "categoria": "cura", "narrativa": "Mujer 57a acude para revision de herida quirurgica limpia y cura programada. Niega dolor y fiebre. TA 126/78, FC 76, FR 16, SatO2 99%, T 36.4."},
    {"id": "S19", "tipo": "sintetico", "categoria": "receta", "narrativa": "Varon 55a solicita receta de medicacion habitual porque se le acabo. Niega sintomas. TA 130/76, FC 74, FR 16, SatO2 98%, T 36.5, Glasgow 15."},
    {"id": "S20", "tipo": "sintetico", "categoria": "administrativo", "narrativa": "Mujer 28a solicita justificante medico para el trabajo. Niega sintomas, dolor y fiebre. TA 116/70, FC 72, FR 15, SatO2 99%, T 36.6."},
]

def cargar_casos_auditoria():
    casos = list(CASOS_SINTETICOS)
    ruta_reales = PROJECT_ROOT / "data" / "interim" / "llm_audit_real_cases_anonymized.csv"
    if ruta_reales.exists():
        reales = pd.read_csv(ruta_reales).dropna(subset=["narrativa"]).head(5)
        casos_reales = []
        for i, row in reales.iterrows():
            casos_reales.append({
                "id": f"R{i + 1:02d}",
                "tipo": "real_anonimizado",
                "categoria": str(row.get("categoria", "real_anonimizado")),
                "narrativa": str(row["narrativa"]),
            })
        casos = casos[: 20 - len(casos_reales)] + casos_reales
    return casos[:20]

casos = cargar_casos_auditoria()
pd.DataFrame(casos)[["id", "tipo", "categoria", "narrativa"]]

,id,tipo,categoria,narrativa
0,S01,sintetico,critico_neuro,Varon 67a. HTA y FA cronica. Tto bisoprolol y ...
1,S02,sintetico,dolor_toracico,Mujer 72a con diabetes e hipertension. Dolor t...
2,S03,sintetico,disnea,Varon 64a EPOC. Consulta por disnea progresiva...
3,S04,sintetico,sepsis,Mujer 81a con insuficiencia renal. Fiebre y de...
4,S05,sintetico,cefalea,"Hombre de unos 40 anos con cefalea brusca, la ..."
5,S06,sintetico,abdomen,Mujer 52a dolor hipocondrio derecho desde la c...
6,S07,sintetico,abdomen_fid,Varon 29a dolor en fosa iliaca derecha de 10 h...
7,S08,sintetico,sincope,Mujer 58a sincope en domicilio con recuperacio...
8,S09,sintetico,trauma,"Varon 35a caida de bicicleta hace 1h, dolor in..."
9,S10,sintetico,alergia,Mujer 24a urticaria generalizada tras comer fr...


## 4. Funciones de trazabilidad

Estas funciones reproducen el flujo del extractor para poder ver el JSON bruto antes de la validacion y normalizacion. Despues se usa el mismo predictor de produccion.

In [4]:
def extraer_json_bruto(narrativa: str, modelo: str = LLM_MODEL) -> str:
    respuesta = ollama.chat(
        model=modelo,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Narrativa:\n{narrativa}"},
        ],
        format=VectorClinico.model_json_schema(),
        options={"temperature": 0.0},
    )
    return respuesta.message.content

def auditar_caso(caso: dict, mostrar: bool = False) -> dict:
    t0 = time.perf_counter()
    narrativa = caso["narrativa"]
    json_bruto = extraer_json_bruto(narrativa)
    vector_llm = VectorClinico.model_validate_json(json_bruto)
    vector_normalizado = normalizar_vector_clinico(vector_llm)
    vector_final = normalizar_vector_clinico(_completar_datos_explicitos(narrativa, vector_normalizado))
    alertas = validar_vector_clinico(vector_final, narrativa)
    adapter_88 = vectorclinico_a_features(vector_final)
    resultado = predictor.predict(vector_final, narrativa)
    latencia = time.perf_counter() - t0

    if mostrar:
        display(Markdown(f"### Caso {caso['id']} - {caso['categoria']}"))
        display(Markdown("**Narrativa original**"))
        display(Markdown(narrativa))
        display(Markdown("**JSON bruto devuelto por el LLM**"))
        print(json.dumps(json.loads(json_bruto), indent=2, ensure_ascii=False))
        display(Markdown("**VectorClinico tras validacion, normalizacion y completado literal**"))
        print(json.dumps(vector_final.model_dump(), indent=2, ensure_ascii=False))
        display(Markdown("**Alertas semanticas del validator**"))
        print(resumen_validacion(alertas))
        display(Markdown("**Adapter: 88 features MIMIC-like**"))
        display(adapter_88.T.rename(columns={0: "valor"}))
        display(Markdown("**Vector final consumido por el modelo congelado**"))
        display(resultado.X.T.rename(columns={0: "valor"}))
        display(Markdown("**Prediccion final**"))
        display(pd.DataFrame({
            "clase": ["ESI 1", "ESI 2", "ESI 3", "ESI 4", "ESI 5"],
            "probabilidad": resultado.probas,
        }))

    return {
        "id": caso["id"],
        "tipo": caso["tipo"],
        "categoria": caso["categoria"],
        "n_sintomas": len(vector_final.sintomas_presentes),
        "sintomas": ", ".join(vector_final.sintomas_presentes),
        "n_alertas_validacion": len(alertas),
        "alertas": resumen_validacion(alertas),
        "esi_predicho": resultado.clase_predicha,
        "confianza": resultado.confianza,
        "p_esi1": float(resultado.probas[0]),
        "alerta_a1": resultado.alerta_a1_activada,
        "latencia_s": latencia,
    }

## 5. Trazado completo de un caso

Se muestra un caso de principio a fin para comprobar visualmente cada paso del pipeline.

In [5]:
_ = auditar_caso(casos[0], mostrar=True)

2026-06-01 19:22:55.578 | INFO     | triaje_ia.data.features:_calcular_missingness_vitales:176 - Missingness vitales calculado
2026-06-01 19:22:55.582 | INFO     | triaje_ia.data.features:_calcular_banderas_vitales:197 - Banderas vitales calculadas
2026-06-01 19:22:55.587 | INFO     | triaje_ia.data.features:_calcular_scores_compuestos:305 - Scores compuestos calculados
2026-06-01 19:22:55.588 | INFO     | triaje_ia.data.features:calcular_features_bloque1:315 - Bloque 1 completado: missingness + banderas + scores
2026-06-01 19:22:55.592 | INFO     | triaje_ia.data.features:calcular_features_bloque2:345 - Bloque 2 completado: demográfico y logístico
2026-06-01 19:22:55.613 | INFO     | triaje_ia.data.features:calcular_features_bloque3:502 - Bloque 3 completado: 18 flags ATC + 3 super-flags
2026-06-01 19:22:55.615 | INFO     | triaje_ia.data.features:calcular_features_bloque4:519 - Bloque 4 completado: polifarmacia
2026-06-01 19:22:55.626 | INFO     | triaje_ia.data.features:calcular_fea

### Caso S01 - critico_neuro

**Narrativa original**

Varon 67a. HTA y FA cronica. Tto bisoprolol y acenocumarol. Traido por familia por bajo nivel de consciencia brusco hace 1h. Sin fiebre. TA 185/110, FC 48 irregular, FR 14, SatO2 94%, T 36.2. GCS 10.

**JSON bruto devuelto por el LLM**

...

,valor
pain,NaN
o2sat,94.0
resprate,14
heartrate,48
dbp,110
...,...
hx_digestivo,0
hx_metabolico_renal,0
hx_infeccioso,0
hx_trauma_muscular,0


**Vector final consumido por el modelo congelado**

,valor
qsofa,0.000000
news2,2.000000
age,67.000000
pain,NaN
n_medicamentos,2.000000
...,...
bert_svd_10,0.529489
bert_svd_11,-1.040929
bert_svd_12,0.688567
bert_svd_13,-0.477359


**Prediccion final**

,clase,probabilidad
0,ESI 1,0.511712
1,ESI 2,0.447798
2,ESI 3,0.038612
3,ESI 4,0.001806
4,ESI 5,0.000072


## 6. Auditoria por lote de 20 historias

Se ejecuta el mismo flujo sobre todos los casos. Si algun caso falla, se registra el error para poder revisarlo sin perder el resto de la auditoria.

In [6]:
filas = []
for caso in casos:
    try:
        filas.append(auditar_caso(caso, mostrar=False))
    except Exception as exc:
        filas.append({
            "id": caso["id"],
            "tipo": caso["tipo"],
            "categoria": caso["categoria"],
            "error": repr(exc),
        })

resultados = pd.DataFrame(filas)
resultados

2026-06-01 19:23:23.035 | INFO     | triaje_ia.data.features:_calcular_missingness_vitales:176 - Missingness vitales calculado
2026-06-01 19:23:23.039 | INFO     | triaje_ia.data.features:_calcular_banderas_vitales:197 - Banderas vitales calculadas
2026-06-01 19:23:23.045 | INFO     | triaje_ia.data.features:_calcular_scores_compuestos:305 - Scores compuestos calculados
2026-06-01 19:23:23.046 | INFO     | triaje_ia.data.features:calcular_features_bloque1:315 - Bloque 1 completado: missingness + banderas + scores
2026-06-01 19:23:23.050 | INFO     | triaje_ia.data.features:calcular_features_bloque2:345 - Bloque 2 completado: demográfico y logístico
2026-06-01 19:23:23.071 | INFO     | triaje_ia.data.features:calcular_features_bloque3:502 - Bloque 3 completado: 18 flags ATC + 3 super-flags
2026-06-01 19:23:23.073 | INFO     | triaje_ia.data.features:calcular_features_bloque4:519 - Bloque 4 completado: polifarmacia
2026-06-01 19:23:23.087 | INFO     | triaje_ia.data.features:calcular_fea

,id,tipo,categoria,n_sintomas,sintomas,n_alertas_validacion,alertas,esi_predicho,confianza,p_esi1,alerta_a1,latencia_s
0,S01,sintetico,critico_neuro,2,"altered mental status, bradycardia",0,Sin alertas de validacion,1,0.511712,0.511712,False,10.006001
1,S02,sintetico,dolor_toracico,2,"chest pain, nausea",0,Sin alertas de validacion,1,0.537625,0.537625,False,8.155737
2,S03,sintetico,disnea,2,"dyspnea, cough",1,"Validación: 0 errores, 0 warnings\n INFO [niv...",1,0.932748,0.932748,False,8.051606
3,S04,sintetico,sepsis,2,"fever, altered mental status",0,Sin alertas de validacion,1,0.869347,0.869347,False,8.095817
4,S05,sintetico,cefalea,2,"thunderclap headache, vomiting",0,Sin alertas de validacion,3,0.438651,0.144271,False,7.706074
5,S06,sintetico,abdomen,2,"right upper quadrant abdominal pain, nausea",0,Sin alertas de validacion,3,0.788405,0.020565,False,7.751843
6,S07,sintetico,abdomen_fid,2,"right lower quadrant abdominal pain, vomiting",0,Sin alertas de validacion,3,0.743549,0.026821,False,7.755891
7,S08,sintetico,sincope,2,"syncope, dizziness",0,Sin alertas de validacion,2,0.463839,0.212036,False,7.663550
8,S09,sintetico,trauma,2,"upper limb pain, wound",0,Sin alertas de validacion,3,0.545137,0.027606,False,7.454529
9,S10,sintetico,alergia,2,"urticaria, stridor",0,Sin alertas de validacion,1,0.819225,0.819225,False,7.464995


## 7. Resumen de seguridad y calidad de extraccion

Esta tabla ayuda a identificar casos con omisiones, alertas de validacion o aviso A1. No se usa para cambiar umbrales ni seleccionar modelo.

In [7]:
columnas_resumen = [
    "id", "tipo", "categoria", "n_sintomas", "sintomas",
    "n_alertas_validacion", "esi_predicho", "confianza", "p_esi1", "alerta_a1",
]
resultados[[c for c in columnas_resumen if c in resultados.columns]].sort_values(
    ["alerta_a1", "n_alertas_validacion", "p_esi1"], ascending=[False, False, False]
)

,id,tipo,categoria,n_sintomas,sintomas,n_alertas_validacion,esi_predicho,confianza,p_esi1,alerta_a1
12,S13,sintetico,convulsion,2,"seizure, altered mental status",0,2,0.471433,0.419227,True
2,S03,sintetico,disnea,2,"dyspnea, cough",1,1,0.932748,0.932748,False
16,S17,sintetico,una_encarnada,2,"toe pain, ingrown toenail",1,3,0.412434,0.097650,False
19,S20,sintetico,administrativo,1,administrative request,1,4,0.359625,0.091488,False
3,S04,sintetico,sepsis,2,"fever, altered mental status",0,1,0.869347,0.869347,False
9,S10,sintetico,alergia,2,"urticaria, stridor",0,1,0.819225,0.819225,False
1,S02,sintetico,dolor_toracico,2,"chest pain, nausea",0,1,0.537625,0.537625,False
0,S01,sintetico,critico_neuro,2,"altered mental status, bradycardia",0,1,0.511712,0.511712,False
11,S12,sintetico,rectorragia,2,"rectal bleeding, weakness",0,2,0.645428,0.325037,False
7,S08,sintetico,sincope,2,"syncope, dizziness",0,2,0.463839,0.212036,False


## 8. Conclusiones

La auditoría confirma que el flujo de producción con `extractor_system_v3_final.txt` funciona de extremo a extremo: el LLM devuelve JSON compatible con `VectorClinico`, la normalización conserva la información clínica relevante, el adapter genera las features esperadas y `TriajePredictor` aplica el modelo final.

En los 20 casos auditados, el extractor mantiene buena fidelidad semántica: diferencia motivos críticos, urgentes, leves y administrativos; conserva las constantes vitales explícitas; normaliza medicación frecuente a clases terapéuticas; y no deja vacíos los motivos leves como receta, cura o justificante. El validator no produce errores ni warnings, solo avisos informativos en casos concretos.

La política de seguridad A1 se comporta como estaba diseñada. En los casos donde la clase argmax ya es ESI 1 no se activa alerta adicional. En el caso de convulsión, el modelo sugiere ESI 2, pero `P(ESI1)=0.419`, por lo que se activa la alerta A1 sin cambiar automáticamente la predicción. Esto refuerza el papel de la alerta como mecanismo visual de seguridad, no como reclasificador.

También aparecen límites importantes. Algunas predicciones deben interpretarse con prudencia: por ejemplo, un caso con cefalea brusca tipo “peor de su vida” queda en ESI 3, lo que sugiere que la extracción puede ser correcta pero el modelo final puede no reflejar toda la gravedad clínica esperada en determinados escenarios sintéticos. Por tanto, esta auditoría valida la trazabilidad técnica del pipeline, pero no constituye validación clínica.

Como conclusión práctica, el prompt v3 final es adecuado para auditar producción: extrae de forma conservadora, mantiene el contrato del esquema y permite inspeccionar cada paso del sistema. Para cerrar el análisis sería recomendable complementar esta auditoría con la sección de información mínima, evaluando qué campos estabilizan más la extracción y la predicción: edad, sexo, motivo, duración, constantes, dolor, antecedentes y medicación relevante.
 